# Athena emulator — data engineering

You're a data engineer running lake ETL on Athena: land parquet datasets on
S3, register them in the Glue catalog, transform with CTAS, add partitions
as data arrives, and export results with UNLOAD. Each step runs against the
live stack and records a measured capability row.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Per ADR-0002, only Athena traffic goes to the emulator; S3/Glue calls must
still land on moto.


In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import time
import uuid

import awswrangler as wr
import boto3
import pandas as pd
from _evidence import persist, record, render
from _helpers import (
    delete_run_buckets,
    delete_run_databases,
    object_keys_under,
)

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb03- prefix (nb03_ for Glue
# databases and tables, where '-' is invalid) so cleanup can discover
# leftovers — its own and any failed run's — instead of tracking a list.
ATHENA_RESOURCE_PREFIX = "nb03-"
DATABASE_PREFIX = "nb03_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}{RUN_SUFFIX}"
scratch_database = f"{DATABASE_PREFIX}scratch_{RUN_SUFFIX}"
orders_table = f"{DATABASE_PREFIX}orders_{RUN_SUFFIX}"
sales_table = f"{DATABASE_PREFIX}sales_{RUN_SUFFIX}"
ctas_table = f"{DATABASE_PREFIX}ctas_{RUN_SUFFIX}"
enc_table = f"{DATABASE_PREFIX}enc_{RUN_SUFFIX}"
external_table = f"{DATABASE_PREFIX}ext_{RUN_SUFFIX}"
meta_table = f"{DATABASE_PREFIX}meta_{RUN_SUFFIX}"
results_prefix = f"s3://{bucket}/results/"


def run_statement(sql: str) -> dict:
    """Submit ``sql`` and poll to a terminal state; return the execution."""
    query_id = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": results_prefix},
    )["QueryExecutionId"]
    for _ in range(60):
        execution = athena.get_query_execution(QueryExecutionId=query_id)[
            "QueryExecution"
        ]
        if execution["Status"]["State"] in (
            "SUCCEEDED",
            "FAILED",
            "CANCELLED",
        ):
            return execution
        time.sleep(0.5)
    raise TimeoutError(f"{query_id} did not reach a terminal state")


print(f"bucket={bucket} database={database}")

bucket=nb03-57f15e792ed7 database=nb03_6a03b22f


## Land the raw datasets

`wr.s3.to_parquet(dataset=True)` writes parquet under the dataset prefix and
registers the table — and its `region=…/` partition dirs — in the Glue
catalog, so Trino resolves it (moto backs the metastore, ADR-0005).


In [3]:
s3.create_bucket(Bucket=bucket)
glue.create_database(DatabaseInput={"Name": database})

orders = pd.DataFrame(
    {"id": [1, 2, 3], "item": ["a", "b", "c"], "amount": [9.5, 1.0, 4.25]}
)
wr.s3.to_parquet(
    df=orders,
    path=f"s3://{bucket}/datasets/orders/",
    dataset=True,
    database=database,
    table=orders_table,
)

sales = pd.DataFrame(
    {"quantity": [1, 2], "amount": [9.5, 4.0], "region": ["EU", "US"]}
)
wr.s3.to_parquet(
    df=sales,
    path=f"s3://{bucket}/datasets/sales/",
    dataset=True,
    database=database,
    table=sales_table,
    partition_cols=["region"],
)

seed_partitions = wr.catalog.get_partitions(
    database=database, table=sales_table
)
assert wr.catalog.does_table_exist(database=database, table=orders_table)
assert sorted(p[0] for p in seed_partitions.values()) == ["EU", "US"]
record(
    "wr.s3.to_parquet(dataset=True) registers tables + partitions",
    "PASS",
    f"{orders_table} + {sales_table} registered; "
    f"sales partitions={list(seed_partitions.values())}",
)

Evidence(feature='wr.s3.to_parquet(dataset=True) registers tables + partitions', status='PASS', detail="nb03_orders_6a03b22f + nb03_sales_6a03b22f registered; sales partitions=[['EU'], ['US']]", latency_ms=None)

## Transform into a named table — CTAS

`wr.athena.create_ctas_table` wraps your SQL in `CREATE TABLE …
WITH(external_location, format='PARQUET') AS …`, leaves the named table
registered in Glue, and returns the execution's `-manifest.csv` location.


In [4]:
started = time.monotonic()
ctas_result = wr.athena.create_ctas_table(
    sql=f'SELECT * FROM "{database}"."{orders_table}" WHERE amount > 2',
    database=database,
    ctas_database=database,
    ctas_table=ctas_table,
    s3_output=f"s3://{bucket}/ctas-out/",
    wait=True,
)
ctas_metadata = ctas_result["ctas_query_metadata"]
data_objects = object_keys_under(s3, bucket, f"ctas-out/{ctas_table}/")
ctas_frame = wr.athena.read_sql_query(
    f'SELECT * FROM "{database}"."{ctas_table}" ORDER BY id',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert ctas_frame["id"].tolist() == [1, 3]
assert data_objects
record(
    "wr.athena.create_ctas_table (named CTAS → Glue table + parquet)",
    "PASS",
    f"{ctas_metadata.execution_id} read back {ctas_frame.shape}; "
    f"{len(data_objects)} object(s) at external_location; "
    f"manifest={ctas_metadata.manifest_location.split('/')[-1]}",
    (time.monotonic() - started) * 1000,
)
ctas_frame

,id,item,amount
0,1,a,9.50
1,3,c,4.25


## Encrypted writes

Two write surfaces take SSE-KMS settings: `create_ctas_table` maps
`encryption`/`kms_key` onto the execution's `EncryptionConfiguration`
(echoed on `GetQueryExecution`); `to_parquet` sends the KMS write headers.


In [5]:
kms_key = (
    "arn:aws:kms:us-east-1:123456789012:key/"
    "00000000-0000-0000-0000-000000000000"
)
enc_result = wr.athena.create_ctas_table(
    sql="SELECT 1 AS n",
    database=database,
    ctas_database=database,
    ctas_table=enc_table,
    s3_output=results_prefix,
    encryption="SSE_KMS",
    kms_key=kms_key,
    wait=True,
)
encryption_echo = enc_result["ctas_query_metadata"].raw_payload[
    "ResultConfiguration"
]["EncryptionConfiguration"]
assert encryption_echo == {
    "EncryptionOption": "SSE_KMS",
    "KmsKey": kms_key,
}
record(
    "SSE_KMS EncryptionConfiguration on CTAS",
    "PASS",
    f"EncryptionConfiguration={encryption_echo}",
)

wr.s3.to_parquet(
    df=orders,
    path=f"s3://{bucket}/kms/orders.parquet",
    dataset=False,
    s3_additional_kwargs={
        "ServerSideEncryption": "aws:kms",
        "SSEKMSKeyId": kms_key,
    },
)
kms_objects = object_keys_under(s3, bucket, "kms/")
assert kms_objects == ["kms/orders.parquet"]
record(
    "wr.s3.to_parquet SSE-KMS write headers",
    "PASS",
    f"objects={kms_objects}",
)

Evidence(feature='wr.s3.to_parquet SSE-KMS write headers', status='PASS', detail="objects=['kms/orders.parquet']", latency_ms=None)

## DDL from the catalog

`wr.athena.generate_create_query` reads the Glue `Table` definition and
emits the `CREATE EXTERNAL TABLE` statement that would recreate it — the
Athena spelling the next section submits.


In [6]:
generated_ddl = wr.athena.generate_create_query(
    table=sales_table, database=database
)
assert "CREATE EXTERNAL TABLE" in generated_ddl
assert "PARTITIONED BY" in generated_ddl
record(
    "generate_create_query emits CREATE EXTERNAL TABLE DDL",
    "PASS",
    f"{len(generated_ddl)} chars; external=True partitioned=True",
)
print(generated_ddl)

CREATE EXTERNAL TABLE `nb03_sales_6a03b22f`(
  `quantity` bigint, 
  `amount` double)
PARTITIONED BY ( 
  `region` string)
ROW FORMAT SERDE 
  'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe' 
STORED AS INPUTFORMAT 
  'org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat' 
OUTPUTFORMAT 
  'org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat'
LOCATION
  's3://nb03-57f15e792ed7/datasets/sales/'
TBLPROPERTIES (
  'classification'='parquet', 
  'compressionType'='snappy', 
  'typeOfData'='file', 
  'projection.enabled'='false')


## Register staged files — Athena DDL

`CREATE EXTERNAL TABLE` and `ALTER TABLE … ADD PARTITION` are Hive
vocabulary Trino's grammar lacks; the emulator's dialect map rewrites them
to `CREATE TABLE … WITH(…)` and `CALL system.register_partition`.


In [7]:
external_location = f"s3://{bucket}/ext/{external_table}/"
staged = pd.DataFrame({"id": [10, 20], "item": ["staged-a", "staged-b"]})
wr.s3.to_parquet(
    df=staged,
    path=f"{external_location}region=EU/part-0.parquet",
    dataset=False,
)

external_execution = run_statement(
    f"CREATE EXTERNAL TABLE {external_table} (id bigint, item string) "
    "PARTITIONED BY (region string) "
    f"STORED AS PARQUET LOCATION '{external_location}'"
)
assert external_execution["Status"]["State"] == "SUCCEEDED"
registered = glue.get_table(DatabaseName=database, Name=external_table)[
    "Table"
]
assert registered["TableType"] == "EXTERNAL_TABLE"
record(
    "CREATE EXTERNAL TABLE via StartQueryExecution (Athena DDL)",
    "PASS",
    f"{external_execution['QueryExecutionId']} SUCCEEDED; "
    f"{external_table} registered EXTERNAL_TABLE",
)

alter_execution = run_statement(
    f"ALTER TABLE {external_table} ADD IF NOT EXISTS "
    f"PARTITION (region='EU') LOCATION '{external_location}region=EU/'"
)
assert alter_execution["Status"]["State"] == "SUCCEEDED"
partition_values = [
    partition["Values"]
    for partition in glue.get_partitions(
        DatabaseName=database, TableName=external_table
    )["Partitions"]
]
assert ["EU"] in partition_values
external_frame = wr.athena.read_sql_query(
    f'SELECT item, region FROM "{database}"."{external_table}" ORDER BY id',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
assert external_frame["item"].tolist() == ["staged-a", "staged-b"]
assert external_frame["region"].tolist() == ["EU", "EU"]
record(
    "ALTER TABLE … ADD PARTITION (Athena DDL)",
    "PASS",
    f"{alter_execution['QueryExecutionId']} SUCCEEDED; "
    f"Glue partitions={partition_values}; "
    f"read back {external_frame.shape}",
)

Evidence(feature='ALTER TABLE … ADD PARTITION (Athena DDL)', status='PASS', detail="c2465b73-6173-4af2-bafe-a0a97774c0cf SUCCEEDED; Glue partitions=[['EU']]; read back (2, 2)", latency_ms=None)

## Discover late-arriving partitions

A file lands under a `region=XX/` dir the catalog never saw. `repair_table`
(`MSCK REPAIR TABLE`, dialect-mapped to `CALL system.sync_partition_metadata`)
registers it; `SHOW PARTITIONS` and `"<t>$partitions"` then list it.


In [8]:
late = pd.DataFrame({"id": [30], "item": ["staged-c"]})
wr.s3.to_parquet(
    df=late,
    path=f"{external_location}region=XX/part-0.parquet",
    dataset=False,
)

repair_state = wr.athena.repair_table(
    table=external_table, database=database, s3_output=results_prefix
)
assert repair_state == "SUCCEEDED"
record(
    "repair_table (MSCK REPAIR TABLE) discovers new partitions",
    "PASS",
    f"final state {repair_state}",
)

shown = run_statement(f"SHOW PARTITIONS {external_table}")
assert shown["Status"]["State"] == "SUCCEEDED"
shown_rows = athena.get_query_results(
    QueryExecutionId=shown["QueryExecutionId"]
)["ResultSet"]["Rows"]
shown_regions = sorted(
    row["Data"][0]["VarCharValue"] for row in shown_rows[1:]
)
assert shown_regions == ["EU", "XX"]
record(
    "SHOW PARTITIONS <table> (Athena spelling)",
    "PASS",
    f"{shown['QueryExecutionId']} → partitions {shown_regions}",
)

partition_frame = wr.athena.read_sql_query(
    f'SELECT region FROM "{database}"."{external_table}$partitions"',
    database=database,
    ctas_approach=False,
    s3_output=results_prefix,
)
listed_regions = sorted(partition_frame["region"].tolist())
assert listed_regions == ["EU", "XX"]
record(
    '"<table>$partitions" virtual table lists Glue partitions',
    "PASS",
    f"Glue partitions={listed_regions}",
)

Evidence(feature='"<table>$partitions" virtual table lists Glue partitions', status='PASS', detail="Glue partitions=['EU', 'XX']", latency_ms=None)

## Catalog partition helpers

Files written outside `dataset=True` never reach the catalog;
`add_parquet_partitions` registers a staged location explicitly and
`delete_partitions` unregisters it without touching the objects.


In [9]:
ap_sales = pd.DataFrame({"quantity": [5], "amount": [2.5], "region": ["AP"]})
ap_location = f"s3://{bucket}/datasets/sales/region=AP/"
wr.s3.to_parquet(
    df=ap_sales, path=f"{ap_location}part-0.parquet", dataset=False
)
wr.catalog.add_parquet_partitions(
    database=database,
    table=sales_table,
    partitions_values={ap_location: ["AP"]},
)
after_add = wr.catalog.get_partitions(database=database, table=sales_table)
assert ["AP"] in after_add.values()
record(
    "wr.catalog.add_parquet_partitions registers a staged partition",
    "PASS",
    f"{len(after_add)} partitions after add: {list(after_add.values())}",
)

wr.catalog.delete_partitions(
    table=sales_table,
    database=database,
    partitions_values=[["AP"]],
)
after_delete = wr.catalog.get_partitions(database=database, table=sales_table)
assert ["AP"] not in after_delete.values()
record(
    "wr.catalog.delete_partitions unregisters a partition",
    "PASS",
    f"{len(after_delete)} partitions after delete: "
    f"{list(after_delete.values())}",
)

Evidence(feature='wr.catalog.delete_partitions unregisters a partition', status='PASS', detail="2 partitions after delete: [['EU'], ['US']]", latency_ms=None)

## Register an existing prefix

`wr.s3.store_parquet_metadata` is the inverse of `dataset=True`: parquet
files already on S3 get their schema inferred and the table + Hive-style
partitions registered in one call — no repair statement needed.


In [10]:
meta_prefix = f"s3://{bucket}/meta/sales/"
wr.s3.to_parquet(
    df=sales, path=meta_prefix, dataset=True, partition_cols=["region"]
)
columns_types, partitions_types, _ = wr.s3.store_parquet_metadata(
    path=meta_prefix,
    database=database,
    table=meta_table,
    dataset=True,
)
meta_partitions = wr.catalog.get_partitions(
    database=database, table=meta_table
)
assert wr.catalog.does_table_exist(database=database, table=meta_table)
record(
    "wr.s3.store_parquet_metadata registers inferred schema + partitions",
    "PASS",
    f"{meta_table} columns={sorted(columns_types)} "
    f"partition_types={partitions_types} "
    f"values={list(meta_partitions.values())}",
)

Evidence(feature='wr.s3.store_parquet_metadata registers inferred schema + partitions', status='PASS', detail="nb03_meta_6a03b22f columns=['amount', 'quantity'] partition_types={'region': 'string'} values=[['EU'], ['US']]", latency_ms=None)

## Export results — UNLOAD

`UNLOAD (sql) TO 's3://…'` lands the result as parquet at the target (the
emulator maps it to a CTAS and drops the generated Glue table when the
statement ends, ADR-0012). Athena rejects a non-empty target — dedicated
prefixes per call.


In [11]:
unload_frame = wr.athena.read_sql_query(
    sql=f'SELECT * FROM "{database}"."{ctas_table}" ORDER BY id',
    database=database,
    ctas_approach=False,
    unload_approach=True,
    s3_output=f"s3://{bucket}/unload-read/",
    unload_parameters={"file_format": "PARQUET"},
)
assert unload_frame["id"].tolist() == [1, 3]
record(
    "read_sql_query unload_approach=True (UNLOAD → parquet read)",
    "PASS",
    f"shape={unload_frame.shape}",
)

unload_meta = wr.athena.unload(
    sql=f'SELECT * FROM "{database}"."{orders_table}" ORDER BY id',
    path=f"s3://{bucket}/export/",
    database=database,
)
export_keys = object_keys_under(s3, bucket, "export/")
assert any(key.endswith("-manifest.csv") for key in export_keys)
assert unload_meta.manifest_location.endswith("-manifest.csv")
record(
    "wr.athena.unload() writes a result set to S3 via UNLOAD",
    "PASS",
    f"{len(export_keys)} object(s) incl. manifest under s3://{bucket}/export/",
)

Evidence(feature='wr.athena.unload() writes a result set to S3 via UNLOAD', status='PASS', detail='3 object(s) incl. manifest under s3://nb03-57f15e792ed7/export/', latency_ms=None)

## Tear down catalog entries

The ETL arc ends by unregistering what it staged: `delete_objects` removes
the prefix, `delete_all_partitions` empties the registry,
`delete_table_if_exists` drops the table, `delete_database` removes an
empty database.


In [12]:
wr.catalog.create_database(scratch_database)

wr.s3.delete_objects(path=meta_prefix)
assert object_keys_under(s3, bucket, "meta/sales/") == []
record(
    "wr.s3.delete_objects removes a staged prefix",
    "PASS",
    f"{meta_prefix} emptied",
)

wr.catalog.delete_all_partitions(table=meta_table, database=database)
remaining_partitions = wr.catalog.get_partitions(
    database=database, table=meta_table
)
assert remaining_partitions == {}
record(
    "wr.catalog.delete_all_partitions empties the registry",
    "PASS",
    f"{meta_table} has 0 partitions",
)

wr.catalog.delete_table_if_exists(database=database, table=meta_table)
assert not wr.catalog.does_table_exist(database=database, table=meta_table)
record(
    "wr.catalog.delete_table_if_exists drops the table",
    "PASS",
    f"{meta_table} absent from the catalog",
)

wr.catalog.delete_database(scratch_database)
remaining_databases = {
    entry["Name"] for entry in glue.get_databases()["DatabaseList"]
}
assert scratch_database not in remaining_databases
record(
    "wr.catalog.delete_database removes an empty database",
    "PASS",
    f"{scratch_database} absent from get_databases",
)

Evidence(feature='wr.catalog.delete_database removes an empty database', status='PASS', detail='nb03_scratch_6a03b22f absent from get_databases', latency_ms=None)

## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/03_data_engineering.md`; `PARITY.md` at the project root is
regenerated from all fragments.


In [13]:
print(render())
persist("03_data_engineering")

| feature | status | detail | latency_ms |
|---|---|---|---|
| wr.s3.to_parquet(dataset=True) registers tables + partitions | PASS | nb03_orders_6a03b22f + nb03_sales_6a03b22f registered; sales partitions=[['EU'], ['US']] |  |
| wr.athena.create_ctas_table (named CTAS → Glue table + parquet) | PASS | 23646d67-da54-4b25-bc1d-f55359c78eab read back (2, 3); 1 object(s) at external_location; manifest=23646d67-da54-4b25-bc1d-f55359c78eab-manifest.csv | 2248.69 |
| SSE_KMS EncryptionConfiguration on CTAS | PASS | EncryptionConfiguration={'EncryptionOption': 'SSE_KMS', 'KmsKey': 'arn:aws:kms:us-east-1:123456789012:key/00000000-0000-0000-0000-000000000000'} |  |
| wr.s3.to_parquet SSE-KMS write headers | PASS | objects=['kms/orders.parquet'] |  |
| generate_create_query emits CREATE EXTERNAL TABLE DDL | PASS | 562 chars; external=True partitioned=True |  |
| CREATE EXTERNAL TABLE via StartQueryExecution (Athena DDL) | PASS | c7d39724-bae9-4c6f-b329-7c50c9596fae SUCCEEDED; nb03_ext_6a03b22f reg

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/03_data_engineering.md')

## Clean up

The run's bucket and Glue databases are throwaway — delete them so the
stack is left as you found it.


In [14]:
delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
delete_run_databases(glue, DATABASE_PREFIX)
print("cleaned: nb03 buckets + glue databases (incl. tables) removed")

cleaned: nb03 buckets + glue databases (incl. tables) removed
